In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../data/host_features_all.csv")
df['time_window'] = pd.to_datetime(df['time_window'])
print(df.shape)
print(df.head())

(97907, 12)
        Source IP         time_window  connection_count  unique_destinations  \
0       1.1.70.73 2017-07-07 03:45:00                 1                    1   
1    1.193.219.24 2017-07-07 04:45:00                 2                    1   
2  101.69.185.208 2017-07-07 04:15:00                 6                    1   
3  101.69.185.240 2017-07-07 04:15:00                 1                    1   
4  101.69.185.240 2017-07-07 04:20:00                 1                    1   

   unique_ports  total_fwd_packets  total_bwd_packets  total_bytes_fwd  \
0             1                2.0                0.0              0.0   
1             2                4.0                0.0            516.0   
2             4               12.0               10.0            264.0   
3             1                2.0                0.0             12.0   
4             1                3.0                0.0             18.0   

   total_bytes_bwd  avg_flow_duration  attack_flow_ratio  \
0 

In [3]:
feature_cols = ['connection_count', 'unique_destinations', 'unique_ports',
                 'total_fwd_packets', 'total_bwd_packets', 
                 'total_bytes_fwd', 'total_bytes_bwd', 'avg_flow_duration']

X = df[feature_cols]

iso = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_score'] = iso.fit_predict(X)
df['anomaly_score_raw'] = iso.decision_function(X)

print(df[['Source IP', 'time_window', 'anomaly_score', 'anomaly_score_raw']].head(20))

          Source IP         time_window  anomaly_score  anomaly_score_raw
0         1.1.70.73 2017-07-07 03:45:00              1           0.307245
1      1.193.219.24 2017-07-07 04:45:00              1           0.288925
2    101.69.185.208 2017-07-07 04:15:00              1           0.220588
3    101.69.185.240 2017-07-07 04:15:00              1           0.306494
4    101.69.185.240 2017-07-07 04:20:00              1           0.304005
5      103.43.91.16 2017-07-07 04:25:00              1           0.298062
6    104.105.77.229 2017-07-07 04:55:00              1           0.289233
7    104.105.77.229 2017-07-07 05:00:00              1           0.302427
8    104.106.241.83 2017-07-07 04:15:00              1           0.293272
9   104.106.249.194 2017-07-07 04:10:00              1           0.287232
10   104.106.252.25 2017-07-07 04:30:00              1           0.298504
11    104.107.3.159 2017-07-07 04:40:00              1           0.298504
12  104.118.220.130 2017-07-07 04:40:0

In [4]:
check = df[df['Source IP'].isin(['172.16.0.1', '192.168.10.50'])]
print(check[['Source IP', 'time_window', 'anomaly_score', 'anomaly_score_raw', 'attack_flow_ratio']])

           Source IP         time_window  anomaly_score  anomaly_score_raw  \
763       172.16.0.1 2017-07-07 03:55:00             -1          -0.201903   
764       172.16.0.1 2017-07-07 04:00:00             -1          -0.203008   
765       172.16.0.1 2017-07-07 04:05:00             -1          -0.203562   
766       172.16.0.1 2017-07-07 04:10:00             -1          -0.203008   
767       172.16.0.1 2017-07-07 04:15:00             -1          -0.188212   
...              ...                 ...            ...                ...   
82551  192.168.10.50 2017-07-05 12:35:00             -1          -0.072765   
82552  192.168.10.50 2017-07-05 12:40:00             -1          -0.062644   
82553  192.168.10.50 2017-07-05 12:45:00             -1          -0.070348   
82554  192.168.10.50 2017-07-05 12:50:00             -1          -0.074627   
82555  192.168.10.50 2017-07-05 12:55:00             -1          -0.052507   

       attack_flow_ratio  
763             1.000000  
764      

In [5]:
df.to_csv("../data/host_features_with_anomaly.csv", index=False)
print("saved:", df.shape)

saved: (97907, 14)
